# Ground Truth Generatie – Dataset A

Handmatige annotatie van lopercentroïden in videoframes van de atletiekpiste.

> **Vereisten:**
> - Google Colab (de annotatie-interface maakt gebruik van Colab-specifieke JavaScript)
> - Video `Atletiekpiste.mp4` (verkrijgbaar via mail; zie `data/README.md`)
>
> **Output:** `../../data/ground_truth/dataset_A/ground_truth_atletiekpiste.json`

### Initialisatie

In [1]:
import cv2
import numpy as np
import base64
import json
import os
from IPython.display import display, Javascript

try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Waarschuwing: de annotatiefunctie vereist Google Colab.")

# Pad naar de videodata (pas aan indien nodig)
VIDEO_PATH = "../../data/video/Atletiekpiste.mp4"
GT_OUTPUT_PATH = "../../data/ground_truth/dataset_A/ground_truth_atletiekpiste.json"

print("Bibliotheken geladen.")

Waarschuwing: de annotatiefunctie vereist Google Colab.
Bibliotheken geladen.


### Annotatiefunctie laden

In [2]:
def annotate_frame_ordered(frame, frame_index, num_runners=8):
    """
    Toont een frame in een interactieve Colab-interface en registreert manueel 
    aangeklikte centroïden van lopers in vaste volgorde.
    
    Parameters
    ----------
    frame : np.ndarray
        BGR-frame uit de video.
    frame_index : int
        Framenummer (voor weergave in de interface).
    num_runners : int
        Aantal te annoteren lopers per frame.
        
    Returns
    -------
    list of [x, y]
        Lijst van pixelcoördinaten. Onzichtbare lopers zijn aangeduid als [-1, -1].
    """
    _, buffer = cv2.imencode('.jpg', frame)
    img_str = base64.b64encode(buffer).decode('utf-8')
    
    js_code = f"""
    async function getOrderedClicks(imgBase64, frameIdx, totalRunners) {{
        const div = document.createElement('div');
        div.innerHTML = `
            <div style="background-color: #f0f0f0; padding: 10px; border-radius: 5px; text-align: center;">
                <h3>Annotatie Frame ${frameIdx}</h3>
                <p>Klik op de lopers in vaste volgorde (1 tot ${totalRunners}).</p>
                <h2 id="instruction_text_${frameIdx}" style="color: blue;">Klik nu op: Loper 1</h2>
                <div style="text-align: center;">
                    <canvas id="canvas_${frameIdx}" style="border:1px solid #000; width: 600px; height: auto; display: block; margin: 0 auto; cursor: crosshair;"></canvas>
                </div>
                <div style="text-align: center; margin-top: 10px;">
                    <button id="btn_skip_${frameIdx}" style="background-color: #ffcc00; margin: 5px; font-size:14px; padding:8px;">
                        Loper onzichtbaar (Skip)
                    </button>
                    <button id="btn_undo_${frameIdx}" style="background-color: #ffcccc; margin: 5px; font-size:14px; padding:8px;">
                        Maak laatste ongedaan
                    </button>
                    <button id="btn_finish_${frameIdx}" disabled style="background-color: #cccccc; margin: 5px; font-size:14px; padding:8px;">
                        Klaar (${totalRunners} geannoteerd)
                    </button>
                    <p id="status_${frameIdx}" style="margin: 5px;">Status: 0 / ${totalRunners} geannoteerd</p>
                </div>
            </div>
        `;
        document.body.appendChild(div);
        
        const canvas = document.getElementById(`canvas_${frameIdx}`);
        const ctx = canvas.getContext('2d');
        const img = new Image();
        let points = [];
        
        return new Promise((resolve) => {{
            img.onload = function() {{
                canvas.width = img.width;
                canvas.height = img.height;
                ctx.drawImage(img, 0, 0);
            }};
            img.src = "data:image/jpeg;base64," + imgBase64;
            
            function updateUI() {{
                ctx.drawImage(img, 0, 0);
                points.forEach((p, index) => {{
                    if (p[0] !== -1) {{
                        ctx.beginPath();
                        ctx.arc(p[0], p[1], 5, 0, 2 * Math.PI, false);
                        ctx.fillStyle = '#00ff00';
                        ctx.fill();
                        ctx.lineWidth = 2;
                        ctx.strokeStyle = '#003300';
                        ctx.stroke();
                        ctx.fillStyle = 'white';
                        ctx.font = "bold 20px Arial";
                        ctx.fillText((index + 1).toString(), p[0] + 8, p[1] - 8);
                    }}
                }});
                
                const nextId = points.length + 1;
                const statusText = document.getElementById(`instruction_text_${frameIdx}`);
                const btnFinish = document.getElementById(`btn_finish_${frameIdx}`);
                const btnSkip = document.getElementById(`btn_skip_${frameIdx}`);
                const canvasEl = document.getElementById(`canvas_${frameIdx}`);
                
                document.getElementById(`status_${frameIdx}`).innerText = `Status: ${points.length} / ${totalRunners} geannoteerd`;
                
                if (points.length < totalRunners) {{
                    statusText.innerText = `Klik nu op: Loper ${nextId}`;
                    statusText.style.color = "blue";
                    btnFinish.disabled = true;
                    btnFinish.style.backgroundColor = "#cccccc";
                    btnSkip.disabled = false;
                    canvasEl.style.pointerEvents = "auto";
                }} else {{
                    statusText.innerText = "Compleet! Klik op 'Klaar' om op te slaan.";
                    statusText.style.color = "green";
                    btnFinish.disabled = false;
                    btnFinish.style.backgroundColor = "#4CAF50";
                    btnFinish.style.color = "white";
                    btnSkip.disabled = true;
                    canvasEl.style.pointerEvents = "none";
                }}
            }}
            
            canvas.onclick = function(e) {{
                if (points.length >= totalRunners) return;
                const rect = canvas.getBoundingClientRect();
                const scaleX = canvas.width / rect.width;
                const scaleY = canvas.height / rect.height;
                const x = (e.clientX - rect.left) * scaleX;
                const y = (e.clientY - rect.top) * scaleY;
                points.push([Math.round(x), Math.round(y)]);
                updateUI();
            }};
            
            document.getElementById(`btn_skip_${frameIdx}`).onclick = function() {{
                if (points.length >= totalRunners) return;
                points.push([-1, -1]);
                updateUI();
            }};
            
            document.getElementById(`btn_undo_${frameIdx}`).onclick = function() {{
                points.pop();
                updateUI();
            }};
            
            document.getElementById(`btn_finish_${frameIdx}`).onclick = function() {{
                div.remove();
                resolve(points);
            }};
        }});
    }}
    """
    display(Javascript(js_code))
    points = output.eval_js(
        f'getOrderedClicks("{img_str}", {frame_index}, {num_runners})'
    )
    print(f"Frame {frame_index}: {len(points)} lopers geannoteerd.")
    return points

print("Annotatiefunctie geladen.")

Annotatiefunctie geladen.


### Annotatie

In [3]:
SKIP_FRAMES = 15
NUM_RUNNERS = 8

cap = cv2.VideoCapture(VIDEO_PATH)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

frames_to_annotate = list(range(0, total_frames - 5, SKIP_FRAMES))

print(f"Video: {total_frames} frames.")
print(f"Te annoteren: {len(frames_to_annotate)} frames (elke {SKIP_FRAMES}e frame).")
print(f"Lopers per frame: {NUM_RUNNERS}\n")
print(f"Instructie: klik lopers altijd in volgorde 1 t/m {NUM_RUNNERS}.")
print("Gebruik de knop 'Skip' voor lopers die niet zichtbaar zijn.")

ground_truth_data = {}
cap = cv2.VideoCapture(VIDEO_PATH)

for i, frame_idx in enumerate(frames_to_annotate):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    
    if ret:
        print(f"\nFrame {frame_idx} ({i + 1}/{len(frames_to_annotate)})...")
        points = annotate_frame_ordered(frame, frame_idx, num_runners=NUM_RUNNERS)
        ground_truth_data[frame_idx] = points
    else:
        print(f"Fout bij lezen frame {frame_idx}.")

cap.release()

# Opslaan als JSON voor gebruik in de evaluatienotebook
os.makedirs(os.path.dirname(GT_OUTPUT_PATH), exist_ok=True)
with open(GT_OUTPUT_PATH, "w") as f:
    json.dump({str(k): v for k, v in ground_truth_data.items()}, f, indent=2)

print(f"\nGround truth opgeslagen: {GT_OUTPUT_PATH}")
print("Format: sleutel = framenummer, waarde = [[x,y], ...]. Onzichtbaar = [-1,-1].")

Video: 0 frames.
Te annoteren: 0 frames (elke 15e frame).
Lopers per frame: 8

Instructie: klik lopers altijd in volgorde 1 t/m 8.
Gebruik de knop 'Skip' voor lopers die niet zichtbaar zijn.

Ground truth opgeslagen: ../../data/ground_truth/dataset_A/ground_truth_atletiekpiste.json
Format: sleutel = framenummer, waarde = [[x,y], ...]. Onzichtbaar = [-1,-1].
